## lets find the parser we need here. 

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import os
from typing import Dict, List, Optional, Literal
from pydantic import BaseModel, Field
from datetime import datetime

load_dotenv(override=True)



/home/zevaz/projects/pianolog-agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
class Activity(BaseModel):
    type: Literal["warmup", "piece", "improvisation", "sight_reading"]

    # repertoire metadata
    piece_name: Optional[str] = None
    composer_name: Optional[str] = None
    key: Optional[str] = None
    section: Optional[str] = None
    bars: Optional[str] = None

    # warmup/exercise metadata
    exercise_name: Optional[str] = None
    tempo: Optional[int] = None
    repetitions: Optional[int] = None

    # general metadata
    focus: Optional[str] = None
    notes: Optional[str] = None


class PracticeSession(BaseModel):
    date: Optional[str] = Field(None, description="YYYY-MM-DD or null")
    duration_min: Optional[int] = Field(None, description="Total duration in minutes or null")
    raw_text: str
    activities: List[Activity]

In [3]:
schema_json = PracticeSession.model_json_schema()
print(schema_json)

{'$defs': {'Activity': {'properties': {'type': {'enum': ['warmup', 'piece', 'improvisation', 'sight_reading'], 'title': 'Type', 'type': 'string'}, 'piece_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Piece Name'}, 'composer_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Composer Name'}, 'key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Key'}, 'section': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Section'}, 'bars': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Bars'}, 'exercise_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Exercise Name'}, 'tempo': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Tempo'}, 'repetitions': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Repetitions'}, 'focus': {'anyOf': [{'type': 'string'},

In [4]:
today = datetime.now().strftime("%Y-%m-%d")
PARSER_PROMPT = f"""
You are a deterministic parser for piano practice logs.

Your task:
- Read the input text.
- Extract structured data.
- Return ONLY valid JSON that matches this schema:

{schema_json}

Additional rules:
- If the date is missing, use today's date: {today}.
- "duration_min" must be an integer number of minutes.
  Convert formats like "~45 min", "50m", "1h20" into minutes.
- Split the text into multiple activities when appropriate.
- "type" must be one of: "warmup", "piece", "improvisation", "sight_reading".
- Do NOT invent activities or fields.
- If a field is not present in the text, set it to null.
- Keep all names and text exactly as written (no normalization).
- "raw_text" must contain the full original input.
- If you cannot identify any activity, return "activities": [] and still include raw_text and date.
- If duration cannot be inferred, set duration_min to null (do not guess).

Output rules:
- Return ONLY valid JSON.
- Do NOT wrap in markdown or code fences.
- Do NOT include explanations or comments.
- Output must start with "{{" and end with "}}".
- Before responding, validate internally that your output is valid JSON and matches the schema exactly.
"""



In [10]:
NORMALIZER_PROMPT = f"""
You are an input normalizer agent for a database that logs piano practice activity.
Your job is to take a structured practice session JSON and normalize it so that all
names, types, keys, bars, sections, and exercises are consistent with the existing
database and follow strict formatting rules.

The input is ALWAYS a JSON object produced by the parser agent: {schema_json}
You MUST return a JSON object with the exact same schema.

Your output MUST follow the exact same schema.

====================================================
NORMALIZATION RULES
====================================================

1. ACTIVITY TYPES
- Valid types: warmup, piece, improvisation, sight_reading.
- Normalize to lowercase.
- If the type is invalid or ambiguous, choose the closest valid one.
- Never invent new types.

2. MUSICAL KEYS
Normalize all key signatures to the format:
"X major" or "X minor"

Rules:
- Capitalize the root note (C, D, E, F, G, A, B).
- Preserve accidentals (# or b).
- Always include a space before “major” or “minor”.
- If multiple keys appear, separate them with ", ".
- If a key appears in the notes field and the key field is null, move it into the key field.

Examples:
- "dbminor" → "Db minor"
- "gmajor" → "G major"
- "f# minor" → "F# minor"
- "D# minor D major" → "D# minor, D major"

3. PIECE AND COMPOSER NAMES (DATABASE-AWARE NORMALIZATION)

3.1 Lowercasing
- Convert all piece and composer names to lowercase.

3.2 Fuzzy Matching Against the Database
You MAY correct misspellings using fuzzy matching against the existing database.
Corrections are allowed ONLY if:
- similarity ≥ 0.85,
- the match is the single closest database entry,
- the correction does not introduce information not present in the database.

If the database contains a unique close match, you MUST normalize to that canonical name.

If similarity is low or ambiguous, keep the original (lowercased).

3.3 Ambiguity Rule
If the normalizer cannot confidently identify a composer or piece, keep the original text.

3.4 Composer–Piece Splitting Rule
If the `piece_name` field contains both a composer and a piece title in the same string,
the normalizer MUST:

1. Attempt to split the composer from the piece title using:
   - common separators: "–", "-", "—", ":", " by "
   - database fuzzy matching (similarity ≥ 0.85)

2. Move the composer into `composer_name`.

3. Keep the remaining text as the normalized `piece_name`.

4. Apply database-aware normalization to both fields.

If the composer cannot be confidently identified (similarity < 0.85 or multiple candidates),
keep the original `piece_name` and leave `composer_name` null.

Examples:
- "chopin – fantasie impromptu, op. 66"
  → composer_name: "chopin"
  → piece_name: "fantasie impromptu, op. 66"

- "Beethovn: sonata no 14"
  → composer_name: "beethoven"
  → piece_name: "sonata no. 14"

- "isabella’s lullaby by takahiro obata"
  → composer_name: "takahiro obata"
  → piece_name: "isabella’s lullaby"

4. EXERCISE NAMES
- Convert to lowercase.
- Normalize common warmup terms:
  scales, arpeggios, octaves, hanon, czerny, mirror scales, weak fingers.
- If multiple exercises appear, separate them with ", ".
- Do not invent new exercises.

5. BARS, SECTIONS, AND LISTS
Bars:
- Normalize all bar lists to comma-separated format.
  Example: "1-4 8 12-16" → "1-4, 8, 12-16"
  Example: "45-50 and 72-80" → "45-50, 72-80"

Sections:
- If multiple sections appear, separate with ", ".

General list rule:
- Any list of items (keys, bars, sections, exercises) must use ", " as the separator.

6. GENRE NORMALIZATION
- Normalize genre names to lowercase.
- Do not infer genres if not present.
- Examples:
  "JAzz" → "jazz"
  "Latin Jazz" → "latin jazz"

7. NOTES FIELD
- Preserve all information.
- Normalize capitalization only when it is clearly non-semantic (e.g., "JAzz" → "jazz").
- Do not move information out of notes unless covered by a specific rule (e.g., key extraction).

8. GENERAL SAFETY RULES
- Do NOT remove information.
- Do NOT invent new information.
- If something cannot be normalized safely, keep the original value.
- Preserve the structure of the input JSON exactly.
- All string fields must remain strings or null.
- All numeric fields must remain numeric or null.

====================
OUTPUT RULES
====================
- Output ONLY valid JSON.
- No markdown, no code fences, no explanations.
- Output must start with "{" and end with "}".
- The output must match the PracticeSession schema exactly.

"""

## Bring the LLMs

In [6]:
rawInput = f"""15 apr 26, 50 min. Warmup: scales on D# minor and D major, weak fingers. 
Worked on Chopin Fantasie Impromptu op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. 
Improv on A minor."""


messages = [{"role": "system", "content": PARSER_PROMPT}] + [{"role": "user", "content": rawInput}]

messages


[{'role': 'system',
  'content': '\nYou are a deterministic parser for piano practice logs.\n\nYour task:\n- Read the input text.\n- Extract structured data.\n- Return ONLY valid JSON that matches this schema:\n\n{\'$defs\': {\'Activity\': {\'properties\': {\'type\': {\'enum\': [\'warmup\', \'piece\', \'improvisation\', \'sight_reading\'], \'title\': \'Type\', \'type\': \'string\'}, \'piece_name\': {\'anyOf\': [{\'type\': \'string\'}, {\'type\': \'null\'}], \'default\': None, \'title\': \'Piece Name\'}, \'composer_name\': {\'anyOf\': [{\'type\': \'string\'}, {\'type\': \'null\'}], \'default\': None, \'title\': \'Composer Name\'}, \'key\': {\'anyOf\': [{\'type\': \'string\'}, {\'type\': \'null\'}], \'default\': None, \'title\': \'Key\'}, \'section\': {\'anyOf\': [{\'type\': \'string\'}, {\'type\': \'null\'}], \'default\': None, \'title\': \'Section\'}, \'bars\': {\'anyOf\': [{\'type\': \'string\'}, {\'type\': \'null\'}], \'default\': None, \'title\': \'Bars\'}, \'exercise_name\': {\'any

In [10]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)



In [11]:
def chat(message, history):
    
    system = PARSER_PROMPT
    messages = [{"role": "system", "content": system}] + [{"role": "user", "content": message}]
    responseGeminy = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=PracticeSession)
    reply = responseGeminy.choices[0].message.parsed

    return reply.model_dump_json(indent=2)

In [13]:
chat( """16 apr 26, 30 min. Warmup: arpeggios in G major, slow tempo around 60 bpm, 5 reps. Focus on evenness.""",'')

'{\n  "date": "2026-04-16",\n  "duration_min": 30,\n  "raw_text": "16 apr 26, 30 min. Warmup: arpeggios in G major, slow tempo around 60 bpm, 5 reps. Focus on evenness.",\n  "activities": [\n    {\n      "type": "warmup",\n      "piece_name": null,\n      "composer_name": null,\n      "key": "G major",\n      "section": null,\n      "bars": null,\n      "exercise_name": "arpeggios",\n      "tempo": 60,\n      "repetitions": 5,\n      "focus": "evenness",\n      "notes": null\n    }\n  ]\n}'

In [8]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Lets string together the parser and normalizer 

In [6]:
rawInput = f"""15 apr 26, 50 min. Warmup: scales on D# minor D major, weak fingers, mirror scales, arpegios. 
Worked on Copin Fantasy Impromp op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. 
Worked on Isabella’s Lullaby by Takahiro Obata. Memorized page 1, tempo 110 bpm
Improv on JAzz over A minor.
reading montunos 6 and 7 by Rebecca mauleon"""

messages = [{"role": "system", "content": PARSER_PROMPT}] + [{"role": "user", "content": rawInput}]



In [ ]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

responseGeminy = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=PracticeSession)
parsedGe = responseGeminy.choices[0].message.parsed

normalized_input_json = parsedGe.model_dump_json(indent=2)


In [11]:
normalizer = OpenAI()
normMessages = [{"role": "system", "content": NORMALIZER_PROMPT}] + [{"role": "user", "content": normalized_input_json}]

resposneNorm = normalizer.beta.chat.completions.parse(model="gpt-4.1-mini", messages=normMessages, response_format=PracticeSession)
norm = resposneNorm.choices[0].message.parsed
print(norm.model_dump_json(indent=2))



{
  "date": "2026-04-15",
  "duration_min": 50,
  "raw_text": "15 apr 26, 50 min. Warmup: scales on D# minor D major, weak fingers, mirror scales, arpegios. Worked on Copin Fantasy Impromp op 66 in C# minor, bars 45-50 and 72-80, focusing on left hand balance, memory and speed. Worked on Isabella’s Lullaby by Takahiro Obata. Memorized page 1, tempo 110 bpm Improv on JAzz over A minor. reading montunos 6 and 7 by Rebecca mauleon",
  "activities": [
    {
      "type": "warmup",
      "piece_name": null,
      "composer_name": null,
      "key": "D# minor, D major",
      "section": null,
      "bars": null,
      "exercise_name": "scales, weak fingers, mirror scales, arpeggios",
      "tempo": null,
      "repetitions": null,
      "focus": "weak fingers",
      "notes": null
    },
    {
      "type": "piece",
      "piece_name": "fantasie impromptu op 66",
      "composer_name": "chopin",
      "key": "C# minor",
      "section": null,
      "bars": "45-50, 72-80",
      "exercise_nam